# Gradient

Run gradient descent, evaluate derivatives and classifier scores, check minimum points, and differentiate matrix expressions.

Run the setup cell first. Each numbered section has editable inputs followed by a
calculation cell. Run that section again after changing its inputs, or use
**Restart Kernel and Run All Cells** to run the examples from a clean state.
Run each workflow's calculation before its result-detail cells. Every input has
a comment describing its format or choices. Optional fields accept `None`,
`"None"`, or blank text. Rerun setup after updating these files.

Matrices accept `1 2; 3 4`, rows on separate lines, fractions such as `1/3`,
nested lists, and NumPy arrays. Rows usually represent samples and columns features.
Results shown with `cache=cache` are saved automatically for use in any notebook
through `cache.load("name")`. The [Cache notebook](09_cache.ipynb) manages them.

[Setup and notebook index](README.md)

In [1]:
from pathlib import Path
import sys

# Works when Jupyter starts in this folder, the repository root, or a subfolder.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "CommonUtils.py").is_file() and (p / "notebooks" / "notebook_tools.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the ee2211-solver repository or its notebooks folder.")
for folder in (PROJECT_ROOT, PROJECT_ROOT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import Markdown, display
import importlib
import notebook_tools
importlib.reload(notebook_tools)
from notebook_tools import (NotebookCache, matrix, optional_matrix, is_blank,
    number, integer, boolean, variable_names, coordinates, choice, show, show_metrics)

pd.set_option("display.precision", 10)
cache = NotebookCache()
print(f"Python {sys.version.split()[0]} | shared cache: {cache.path}")

Python 3.11.17 | shared cache: D:\NUS\Y2S1\EE2211\solvers\ee2211-solver\notebooks\.cache\matrices.sqlite3


## 1. Gradient descent
Variables are inferred alphabetically. Initial values follow that order.
Each update uses all derivatives at the previous point simultaneously.
Tolerance, when supplied, stops when the largest parameter change is small enough.
The history includes iteration zero and derivatives at each displayed point.

In [2]:
cost = 'x**2 + x*y**2'  # Scalar expression; * multiplies, ** or ^ raises powers.
start = [3, 2]  # Coordinates in alphabetical variable order; list or 3,2 string.
rate = 0.2  # Positive learning rate, e.g. 0.2.
iterations = 10  # Nonnegative whole number of updates.
tolerance = None  # Optional nonnegative stopping threshold; None disables.

In [3]:
from GradientDescent import gradient_descent, infer_variables

variables = infer_variables(cost)
print("Variable order:", ", ".join(variables))
expression, gradients, gradient_history = gradient_descent(cost, variables,
    coordinates(start), number(rate), integer(iterations), number(tolerance, optional=True))
display(sp.Eq(sp.Symbol("C"), expression, evaluate=False))
for parameter, derivative in zip(variables, gradients):
    display(Markdown(f"Derivative with respect to **{parameter}**"))
    display(derivative)
history_rows = []
for step in gradient_history:
    history_rows.append({"iteration": step["iteration"], **dict(zip(variables, step["values"])),
        **{f"dC/d{v}": g for v, g in zip(variables, step["gradient"])}, "cost": step["cost"]})
display(pd.DataFrame(history_rows))
show("final_parameters", gradient_history[-1]["values"].reshape(-1, 1), cache=cache, source="Gradient descent")

Variable order: x, y


Eq(C, x**2 + x*y**2)

Derivative with respect to **x**

2*x + y**2

Derivative with respect to **y**

2*x*y

,iteration,x,y,dC/dx,dC/dy,cost
0,0,3.0000000000,2.0000000000,10.0000000000,12.0000000000,21.0000000000
1,1,1.0000000000,-0.4000000000,2.1600000000,-0.8000000000,1.1600000000
2,2,0.5680000000,-0.2400000000,1.1936000000,-0.2726400000,0.3553408000
3,3,0.3292800000,-0.1854720000,0.6929598628,-0.1221444403,0.1197525052
4,4,0.1906880274,-0.1610431119,0.4073109388,-0.0614179867,0.0413073957
5,5,0.1092258397,-0.1487595146,0.2405810726,-0.0324967658,0.0143473856
6,6,0.0611096252,-0.1422601614,0.1424572039,-0.0173869303,0.0049711200
7,7,0.0326181844,-0.1387827754,0.0844970275,-0.0090536843,0.0016921937
8,8,0.0157187789,-0.1369720385,0.0501988971,-0.0043060664,0.0005419854
9,9,0.0056789995,-0.1361108252,0.0298841557,-0.0015459466,0.0001374611


**final_parameters**

,0
0,-0.0002978317
1,-0.1358016359


## 2. Functions and derivatives at a point
Separate vector outputs with semicolons or new lines. Set `wrt`
to an explicit list to include unused inputs in the Jacobian, or `None` to infer
alphabetically. For constants, use an empty coordinate list.

Interpretation can be `Numeric values`, `Sign classification` (one scalar output),
or `Class scores` (largest output, first wins ties, zero-based index). Class
scores need not be probabilities. A scalar output gives a gradient and Hessian;
vector outputs give a Jacobian with output rows and input columns.

In [ ]:
outputs = 'x1**2 + 3*x1*x2 + 2*x2**2'  # Scalar expression or ; separated vector outputs.
wrt = None  # Variable list or comma-separated names; None infers order.
point = [1, -1]  # Coordinates in that order; list or comma-separated numbers.
interpretation = 'Numeric values'  # Numeric values | Sign classification | Class scores.

In [ ]:
from FunctionUtils import analyze_functions

function = analyze_functions(outputs, coordinates(point), variable_names(wrt))
print("Variable order:", function["variables"])
show("function_value", function["value"], cache=cache, source="Function evaluation")
if interpretation == "Sign classification":
    if function["value"].size != 1:
        raise ValueError("Sign classification needs exactly one scalar output.")
    show("function_class", [[int(np.sign(function["value"].item()))]], cache=cache, source="Score sign")
    score = function["expressions"][0]
    display(Markdown("**Decision boundary**"))
    display(sp.Eq(score, 0, evaluate=False))
    if not score.free_symbols:
        print("The score is zero everywhere." if score == 0 else "The constant score has no zero boundary.")
    else:
        last_variable = sorted(score.free_symbols, key=lambda item: item.name)[-1]
        if sp.diff(score, last_variable, 2) == 0 and sp.diff(score, last_variable) != 0:
            solutions = sp.solve(score, last_variable)
            if len(solutions) == 1:
                display(sp.Eq(last_variable, solutions[0], evaluate=False))
elif interpretation == "Class scores":
    show("function_class", [[int(np.argmax(function["value"]))]], cache=cache, source="Largest score")
elif interpretation != "Numeric values":
    raise ValueError("Choose Numeric values, Sign classification, or Class scores.")
if function["variables"]:
    if function["gradient"] is not None:
        show("gradient", function["jacobian_at_point"].T, cache=cache, source="Point derivatives")
        display(function["gradient"])
    else:
        show("jacobian", function["jacobian_at_point"], cache=cache, source="Point derivatives")
        display(function["jacobian"])
    if function["hessian"] is not None:
        display(function["hessian"])
        show("hessian_at_point", function["hessian_at_point"])
display(pd.DataFrame({"Expression": [str(e) for e in function["expressions"]], "Dependence on inputs": function["dependence"]}))

### Worked examples: vector derivatives and supplied scores
These examples keep variable order explicit and show both classifier conventions.

In [ ]:
vector_function = analyze_functions("x1**2+x2; x1*x2", [2, -1, 7], ["x1", "x2", "x3"])
show("vector_jacobian", vector_function["jacobian_at_point"])
sign_function = analyze_functions("1-2*x1+x1*x2", [2, 1], ["x1", "x2"])
print("Sign classification:", int(np.sign(sign_function["value"].item())))
display(sp.Eq(sign_function["expressions"][0], 0, evaluate=False))
class_scores = analyze_functions("-0.2; -0.8; -0.1", [])
print("Class-score prediction (zero-based):", int(np.argmax(class_scores["value"])))

## 3. Check a minimum point
This is a check of a supplied unconstrained smooth interior point. Zero gradient
alone does not prove a minimum. A singular multivariable Hessian may be
inconclusive. Global results require the calculator's convex quadratic certificate.
Univariate higher-derivative checks stop at order 12.

Point coordinates may be exact strings such as `"1/3"` or `"pi/2"`.
Try `x^3` at `["0"]` for a stationary point that is neither minimum nor maximum,
or `x^2-y^4` at `["0", "0"]` for an inconclusive Hessian check.

In [ ]:
f = '(x-3)^2+2'  # Scalar function, e.g. (x-3)^2+2.
min_wrt = None  # Variable list or comma-separated names; None infers order.
candidate = ['3']  # Exact coordinates, e.g. ["1/3"] or "pi/2,0".

In [ ]:
from FunctionUtils import check_minimum

minimum = check_minimum(f, coordinates(candidate, exact=True), variable_names(min_wrt))
show_metrics({key: minimum[key] for key in ("classification", "is_minimum", "value", "reason", "eigenvalues")})
show("minimum_value", [[minimum["value"]]], cache=cache, source="Function at candidate point")
display(Markdown("**Symbolic gradient and Hessian**"))
display(minimum["gradient"], minimum["hessian"])
for key in ("gradient_at_point", "hessian_at_point"):
    show("minimum_" + key, minimum[key], cache=cache, source="Minimum derivative check")

## 4. Differentiate a matrix expression
Use the website's real MATLAB-style syntax. Scalar outputs give a gradient with
the variable's shape; matrix outputs give a Jacobian. Input and output entries
are flattened column by column for Jacobians and Hessians.

Definitions can assign numeric matrices or declare symbolic matrices, such as
`syms A [2 2] real`. Leave `at` empty for symbolic results.
Examples include `x'*A*x`, `norm(X*w-y)^2 + lambda*(w'*w)`, `trace(W'*W)`,
and `sum((X*W-Y).^2,'all')`. Arithmetic includes `*`, `/`, `\`, `^`, `.*`,
`./`, and `.^`; both `'` and `.'` transpose real matrices. Functions include
`trace`, `norm`, `sum`, `det`, `inv`, `diag`, `transpose`, `eye`, `zeros`,
`ones`, `sin`, `cos`, `tan`, `exp`, `log`, `sqrt`, and `abs`.
Matrix norms require `'fro'`; matrix division requires square invertible operands.
Indexing is one-based. This is the calculator's supported subset of MATLAB syntax.

In [ ]:
expr = "x'*A*x"  # Real MATLAB-style expression, e.g. "x' * A * x".
definitions = 'A = [1 2; 0 3]\nsyms x [2 1] real'  # Numeric assignments / syms declarations; separate by newline.
variable = 'x'  # Name of the scalar/vector/matrix to differentiate.
at = 'x = [1; 1]'  # Optional assignments, e.g. "x=[1;1]"; None keeps symbolic.

In [ ]:
from MatrixCalculus import differentiate_matrix

matrix_derivative = differentiate_matrix(expr, definitions, variable, "" if is_blank(at) else at)
derivative = matrix_derivative["gradient"] if matrix_derivative["gradient"] is not None else matrix_derivative["jacobian"]
print("Variable entry order:", ", ".join(matrix_derivative["variables"]))
display(Markdown("**Gradient / Jacobian**"))
display(derivative)
show("matrix_derivative", matrix_derivative["derivative_at_point"], cache=cache, source="Matrix differentiation")
display(Markdown("**Expanded expression**"))
display(matrix_derivative["value"])
show("matrix_expression_value", matrix_derivative["value_at_point"], cache=cache, source="Matrix expression")
if matrix_derivative["hessian"] is not None:
    display(Markdown("**Hessian**"))
    display(matrix_derivative["hessian"])
    show("matrix_hessian_at_point", matrix_derivative["hessian_at_point"])
if not is_blank(at) and matrix_derivative["derivative_at_point"] is None:
    print("Some symbols still have no numeric value; the derivative remains symbolic.")